# Объединение парных ридов pRESTO — все лёгкие цепи мыши

Для обеих сохраняемых веток — с найденным IGL-праймером и без совпадения с праймером — используется `AssemblePairs.py align`. R2 задаётся как head, R1 — как tail с обратным комплементированием. Для каждой ветки сохраняются pass/fail-результаты, а успешно объединённые риды собираются вместе для QC.


In [ ]:
from pathlib import Path
import gzip, json, os, shutil, subprocess, time
REPO=Path.cwd().resolve()
if not (REPO/'notebooks').is_dir(): REPO=REPO.parent
VOLUME=Path(os.environ.get('BCR_VOLUME','/data/user/epishkin'))
if not (VOLUME/'raw').is_dir(): VOLUME=REPO
ENV=Path(os.environ.get('BCR_ENV','/opt/conda/envs/bcr_env'))
if not ENV.is_dir(): ENV=Path('/Users/epishkin/mamba/envs/bcr_env')
RUN='SRR32426580'; RAW=VOLUME/'raw'/'PRJNA1226555'
ROOT=VOLUME/'results'/'PRJNA1226555'/'branches'/'legacy_qtrim_min250'
R1=RAW/f'{RUN}_1.fastq.gz'; R2=RAW/f'{RUN}_2.fastq.gz'
def tool(name):
    p=ENV/'bin'/name
    if p.is_file(): return p
    q=shutil.which(name)
    if q: return Path(q)
    raise FileNotFoundError(name)
def fqcount(path):
    with gzip.open(path,'rt') as h: n=sum(1 for _ in h)
    assert n%4==0,path
    return n//4
def run(cmd,out,err,outputs=(),heartbeat=30):
    out=Path(out); err=Path(err); out.parent.mkdir(parents=True,exist_ok=True)
    started=time.monotonic()
    with out.open('w') as o,err.open('w') as e:
        p=subprocess.Popen([str(x) for x in cmd],stdout=o,stderr=e,text=True)
        print(f'PID={p.pid}',flush=True)
        while p.poll() is None:
            sizes=' '.join(f'{Path(x).name}={Path(x).stat().st_size/1e6:.1f}MB' for x in outputs if Path(x).exists())
            print(f'PID={p.pid} elapsed={(time.monotonic()-started)/60:.1f}min {sizes}',flush=True)
            time.sleep(heartbeat)
    if p.returncode: raise RuntimeError(f'rc={p.returncode}; see {err}')
    print(f'DONE elapsed={(time.monotonic()-started)/60:.1f}min',flush=True)
for p in (R1,R2): assert p.is_file(),p
print('ROOT',ROOT)


In [ ]:
SYNC=ROOT/'pr_trimmed'/'sync'; BASE=ROOT/'merged'; LOG=BASE/'logs'
BASE.mkdir(parents=True,exist_ok=True); LOG.mkdir(parents=True,exist_ok=True)
def branch_files(branch):
    fs=sorted((SYNC/branch).glob('*pair-pass*.fastq.gz')); assert len(fs)==2,fs
    return next(p for p in fs if '-1_pair-pass' in p.name),next(p for p in fs if '-2_pair-pass' in p.name)
def assemble(branch,copy_primer=False):
    r1,r2=branch_files(branch); out=BASE/branch; out.mkdir(parents=True,exist_ok=True)
    stdout=LOG/f'{branch}.stdout.log'; stderr=LOG/f'{branch}.stderr.log'
    existing_pass=list(out.glob('*assemble-pass.fastq.gz')); existing_fail=list(out.glob('*assemble-fail.fastq.gz'))
    if len(existing_pass)==1 and len(existing_fail)==2 and stdout.exists() and 'END> AssemblePairs' in stdout.read_text(errors='replace'):
        assert fqcount(existing_fail[0])==fqcount(existing_fail[1])
        print('REUSE_COMPLETE',branch)
        return existing_pass[0],existing_fail[0]
    shutil.rmtree(out); out.mkdir()
    cmd=[tool('AssemblePairs.py'),'align','-1',r2,'-2',r1,'--coord','sra','--rc','tail','--nproc','4','--failed','--gzip-output','--outdir',out,'--outname',f'{RUN}_{branch}']
    if copy_primer: cmd += ['--2f','PRIMER']
    run(cmd,stdout,stderr,list(out.glob('*')))
    ps=list(out.glob('*assemble-pass.fastq.gz')); fs=list(out.glob('*assemble-fail.fastq.gz'))
    assert len(ps)==1 and len(fs)==2,(ps,fs)
    assert fqcount(fs[0])==fqcount(fs[1])
    assert stdout.exists() and 'END> AssemblePairs' in stdout.read_text(errors='replace')
    return ps[0],fs[0]
igl_pass,igl_fail=assemble('igl_primer_pass',True)
unmatched_pass,unmatched_fail=assemble('primer_unmatched',False)
print('IGL',fqcount(igl_pass),fqcount(igl_fail)); print('UNMATCHED',fqcount(unmatched_pass),fqcount(unmatched_fail))


In [ ]:
combined=BASE/f'{RUN}_all_assemble-pass.fastq.gz'
with gzip.open(combined,'wb',compresslevel=1) as out:
    for src in (igl_pass,unmatched_pass):
        with gzip.open(src,'rb') as inp: shutil.copyfileobj(inp,out)
assembled=fqcount(combined); failed=fqcount(igl_fail)+fqcount(unmatched_fail); total=assembled+failed
assert assembled==fqcount(igl_pass)+fqcount(unmatched_pass)
summary={'input_pairs':total,'assembled_pairs':assembled,'failed_pairs':failed,'merge_rate':assembled/total,'igl_primer_pass_assembled':fqcount(igl_pass),'primer_unmatched_assembled':fqcount(unmatched_pass)}
(BASE/'assembly_summary.json').write_text(json.dumps(summary,indent=2)); print(summary)
